## Propose rules for one completed SOURCE run

Run the notebook from the top to create a new proposal batch. Review individual proposals after the batch is stored.

Use `delinquency_pilot = ON` with `balance_pilot = OFF` to propose the utilization-to-delinquency dependency from measured SOURCE bucket rates. Use a fresh batch and review the proposal individually.


In [ ]:
dbutils.widgets.text("source_run_id", "", "Completed SOURCE run ID")
dbutils.widgets.dropdown("balance_pilot", "OFF", ["OFF", "ON"], "Balance pilot")
dbutils.widgets.dropdown("delinquency_pilot", "OFF", ["OFF", "ON"], "Utilization / delinquency pilot")

dbutils.widgets.dropdown("workflow_mode", "OFF", ["OFF", "ON"])
dbutils.widgets.text("reuse_batch_id", "")
workflow_mode = dbutils.widgets.get("workflow_mode") == "ON"
dbutils.widgets.text("workflow_rule_scope", "BALANCE_PILOT")


In [ ]:
import sys
from pathlib import Path

# Keep notebooks/ and src/ together; no package installation is needed.
repo_dirs = (Path.cwd(), Path.cwd().parent, *(Path(path) for path in sys.path if path))
for repo_dir in repo_dirs:
    src_dir = repo_dir if repo_dir.name == "src" else repo_dir / "src"
    if (src_dir / "synthetic_banking" / "config.py").is_file():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("Keep notebooks/ and src/ together, or add the project's src/ folder to the Python path.")

from synthetic_banking.config import PROFILE_SCHEMA, configured_sql
from synthetic_banking.proposals import (
    prepare_proposal_tables, start_proposal_batch, build_column_context,
    build_model_request, call_proposal_model, store_proposals,
)

if workflow_mode:
    from synthetic_banking.workflow import require_source, require_batch, publish_values
    source_run_id = require_source(spark, dbutils.widgets.get("source_run_id").strip())
    reused_batch = dbutils.widgets.get("reuse_batch_id").strip()
    if reused_batch:
        require_batch(spark, source_run_id, reused_batch)
        publish_values(dbutils, source_run_id=source_run_id, batch_id=reused_batch)
        dbutils.notebook.exit("Reusing the explicit stored proposal batch; no new model call.")

prepare_proposal_tables(spark)

In [ ]:
if workflow_mode:
    workflow_scope = dbutils.widgets.get("workflow_rule_scope")
    if workflow_scope not in {"BALANCE_PILOT", "DELINQUENCY_PILOT", "ACCOUNT_CORE"}:
        raise ValueError("Unknown workflow rule scope.")
else:
    workflow_scope = None

source_run_id, batch_id, scope_name = start_proposal_batch(
    spark, dbutils.widgets.get("source_run_id").strip(),
    balance_pilot=(workflow_scope == "BALANCE_PILOT") if workflow_mode else dbutils.widgets.get("balance_pilot") == "ON",
    delinquency_pilot=(workflow_scope == "DELINQUENCY_PILOT") if workflow_mode else dbutils.widgets.get("delinquency_pilot") == "ON",
)
print(f"SOURCE {source_run_id}; proposal batch {batch_id}; scope {scope_name}")

In [ ]:
build_column_context(spark)

Preview the aggregate column context used to build this batch's prompt.

In [ ]:
display(spark.sql(configured_sql("""
SELECT *
FROM ai_column_context
ORDER BY logical_table_name, column_name;
""")))

The column summary above is tied to this batch and its selected SOURCE run. The next cell adds measured dependency evidence to the model prompt.

In [ ]:
from synthetic_banking.proposals import build_model_request

request = build_model_request(spark)

In [ ]:
display(spark.sql(configured_sql("""
SELECT source_run_id, batch_id, scope_name, sha2(prompt, 256) AS prompt_sha256
FROM ai_model_request;
""")))

In [ ]:
call_proposal_model(spark, request)

In [ ]:
display(spark.sql(configured_sql("""
SELECT b.batch_id, b.source_run_id, b.scope_name, b.created_at, b.model_endpoint, b.prompt_sha256, b.response
FROM workspace.synthetic_profile.ai_rule_batches b
JOIN ai_model_request r
    ON b.batch_id = r.batch_id AND b.source_run_id = r.source_run_id;
""")))

Responses are parsed against the fixed rule schema, then stored as PENDING proposals. Rerunning the storage cell does not create duplicates for this batch.

In [ ]:
store_proposals(spark)

In [ ]:
display(spark.sql(configured_sql("""
SELECT
    p.source_run_id,
    p.batch_id,
    proposal_id,
    target_table,
    target_column,
    rule_type,
    parent_columns,
    confidence,
    evidence_type,
    statistical_evidence,
    proposed_rule,
    rationale,
    requires_sme_review,
    approval_status,
    reviewed_by,
    reviewed_at

FROM workspace.synthetic_profile.ai_rule_proposals p
JOIN ai_model_request r
    ON p.batch_id = r.batch_id AND p.source_run_id = r.source_run_id

ORDER BY
    target_table,
    target_column;
""")))

### Review one proposal

Inspect the proposal and its batch before making a decision. Complete any required SME review. To record a decision, run the following statement separately in the Databricks SQL editor. Supply the exact SOURCE, batch, and proposal IDs and choose APPROVED or REJECTED.

```sql
UPDATE workspace.synthetic_profile.ai_rule_proposals
SET approval_status = :review_decision,
    reviewed_by = current_user(),
    reviewed_at = current_timestamp()
WHERE source_run_id = :source_run_id
  AND batch_id = :batch_id
  AND proposal_id = :proposal_id
  AND approval_status = 'PENDING'
  AND reviewed_at IS NULL
  AND :review_decision IN ('APPROVED', 'REJECTED');
```

Approval records a proposal review. Implementation and execution are separate decisions. Legacy rows have no batch ID and are ineligible for review through this workflow. Re-propose them in a new batch if needed.

Provenance: completed SOURCE run → proposal batch → proposal ID → review status, reviewer, and review time.

In [ ]:
if workflow_mode:
    from synthetic_banking.workflow import require_batch, publish_values
    if request["source_run_id"] != source_run_id or request["batch_id"] != batch_id:
        raise ValueError("Model request and stored batch handoff differ.")
    require_batch(spark, source_run_id, batch_id)
    publish_values(dbutils, source_run_id=source_run_id, batch_id=batch_id)